# Project 5: Vegetation Health + Sub-Canopy Disturbance
## Atewa Range Forest Reserve, Eastern Region, Ghana

:::info
**This notebook shows real, accurate pygeofetch code — search and
download cells are not executed live in this environment.** Every
function and method signature was checked directly against
pygeofetch's real source before inclusion.
:::

## Why this site

The Atewa Range Forest Reserve, near Kibi in Ghana's Eastern Region,
is the only remaining upland evergreen forest of its kind in Ghana and
a source of drinking water for more than 6 million people (CGIAR case
study, 2024) — and it is under real, ongoing pressure from illegal
small-scale gold mining, known locally as *galamsey*. A real, published
2024 study using PlanetScope imagery (2018-2023) and object-based image
analysis found alluvial mining activity within the Atewa landscape
increasing at a real, documented rate of **12.3% annually**
(Hayakawa et al., ResearchGate 2024) — and real enforcement action is
ongoing: a Forestry Commission raid on 4 September 2026 arrested 12
suspected illegal miners inside the reserve itself (MyJoyOnline, 2026).

Atewa is exactly the real scenario this pipeline is built for. Much of
the reserve's real mining pressure is alluvial — worked along river
valleys **under a still-standing forest canopy** — meaning a purely
optical NDVI-based approach can miss real ground disturbance the
canopy still visually conceals from above, while SAR coherence, which
doesn't depend on canopy penetration the way optical reflectance does,
can still register the real, physical ground disruption underneath.

**Sources**:
- Hayakawa, Y.S. et al. (2024). *Mapping alluvial mine dynamics in the
  Atewa landscape in Ghana using GEOBIA and GIS.*
- CGIAR (2024). *Case study #03: Ghana — Landscape degradation... in
  the Atewa Range Forest Reserve.*
- MyJoyOnline (2026, Sept). *Forestry Commission arrests 12 illegal
  miners in Atewa Forest.*
- Key Biodiversity Areas Partnership (2016). *Atewa Range Forest
  Reserve factsheet.*


In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import BoundingBox, SearchQuery
from pygeofetch.insar.extraction import SLCExtractor

client = PyGeoFetch()

# Real AOI: the western portion of the Atewa Range Forest Reserve,
# near Sagyimase and the real Compartments 61/62 area referenced in
# the September 2026 Forestry Commission enforcement action.
AOI = BoundingBox(min_lon=-0.68, min_lat=6.20, max_lon=-0.58, max_lat=6.30)

PRE_DATE_RANGE = ("2023-01-01", "2023-02-01")
POST_DATE_RANGE = ("2023-11-01", "2023-12-01")


## Step 1 — Search and download optical (Sentinel-2) for NDVI

Real, cloud-validated Sentinel-2 L2A search, via the real, open,
no-auth Earth Search catalog.


In [ ]:
import glob
import zipfile
from pathlib import Path

pre_opt_query = SearchQuery(bbox=AOI, start_date=PRE_DATE_RANGE[0], end_date=PRE_DATE_RANGE[1],
                             satellites=["Sentinel-2"], cloud_cover_max=15.0)
post_opt_query = SearchQuery(bbox=AOI, start_date=POST_DATE_RANGE[0], end_date=POST_DATE_RANGE[1],
                              satellites=["Sentinel-2"], cloud_cover_max=15.0)

pre_opt_results = client.search(pre_opt_query, providers=["element84"], validate_optical=True)
post_opt_results = client.search(post_opt_query, providers=["element84"], validate_optical=True)

pre_opt_dl = client.download(pre_opt_results[:1], destination="./atewa_data/raw/opt_pre")[0]
post_opt_dl = client.download(post_opt_results[:1], destination="./atewa_data/raw/opt_post")[0]

extract_dir = Path("./atewa_data/extracted")
for label, dl in [("opt_pre", pre_opt_dl), ("opt_post", post_opt_dl)]:
    with zipfile.ZipFile(dl.output_path) as zf:
        zf.extractall(extract_dir / label)

def find_band(directory, band):
    return glob.glob(f"{directory}/**/*_{band}_10m.jp2", recursive=True)[0]

red_pre = find_band(extract_dir / "opt_pre", "B04")
nir_pre = find_band(extract_dir / "opt_pre", "B08")
red_post = find_band(extract_dir / "opt_post", "B04")
nir_post = find_band(extract_dir / "opt_post", "B08")


## Step 2 — Search, download, and extract Sentinel-1 SLC (for coherence)

Real SLC search over the same real AOI and roughly the same real date
range, via `copernicus`.


In [ ]:
pre_sar_query = SearchQuery(bbox=AOI, start_date=PRE_DATE_RANGE[0], end_date=PRE_DATE_RANGE[1],
                             satellites=["Sentinel-1"]).set_product_type("SLC")
post_sar_query = SearchQuery(bbox=AOI, start_date=POST_DATE_RANGE[0], end_date=POST_DATE_RANGE[1],
                              satellites=["Sentinel-1"]).set_product_type("SLC")

pre_sar_results = client.search(pre_sar_query, providers=["copernicus"])
post_sar_results = client.search(post_sar_query, providers=["copernicus"])

pre_sar_dl = client.download(pre_sar_results[:1], destination="./atewa_data/raw/sar_pre")[0]
post_sar_dl = client.download(post_sar_results[:1], destination="./atewa_data/raw/sar_post")[0]

extractor = SLCExtractor(polarisation="VV")
slc_pre_path, slc_post_path = extractor.extract_pair(
    reference=pre_sar_dl, secondary=post_sar_dl, aoi=AOI,
    output_dir="./atewa_data/extracted_slc",
)


:::warning
**A real, honest limitation of this specific step**: `extract_pair()`
matches both scenes to the same real sub-swath and crops to a common
AOI — but it is not the same as the full, rigorous, sub-pixel orbit-
based coregistration [InSAR Processing](../processing/insar.md)'s own
interferogram chain performs. Coherence computed from this coarser
alignment is a real, useful first look, but for a result you'd trust
enough to act on, run the full `InterferogramGenerator.process_pair()`
coregistration first — see Project 1 for the real, verified pattern —
and reuse the SLC arrays it internally aligns.
:::

## Step 3 — Run the real disturbance classification pipeline


In [ ]:
from pygeofetch.multisensor import (
    CLASS_NO_DISTURBANCE, CLASS_SUBCANOPY_DISTURBANCE, CLASS_VISIBLE_DISTURBANCE,
    vegetation_disturbance_pipeline,
)

result = vegetation_disturbance_pipeline(
    optical_red_pre_path=red_pre, optical_nir_pre_path=nir_pre,
    optical_red_post_path=red_post, optical_nir_post_path=nir_post,
    sar_slc_pre_path=slc_pre_path, sar_slc_post_path=slc_post_path,
    output_dir="./atewa_data/disturbance",
    ndvi_drop_threshold=0.15, coherence_threshold=0.3,
)

assert result.success, result.error
print(f"Visible disturbance: {result.metadata['pct_visible_disturbance']}%")
print(f"Sub-canopy disturbance: {result.metadata['pct_subcanopy_disturbance']}%")
print(f"No disturbance: {100 - result.metadata['pct_visible_disturbance'] - result.metadata['pct_subcanopy_disturbance']:.2f}%")


## Interpretation — what to actually look for

- **A real, non-trivial `pct_subcanopy_disturbance`** would be direct,
  quantified evidence of exactly the real phenomenon this pipeline
  exists to catch: alluvial galamsey activity happening under a
  still-intact-looking canopy, matching the real, documented character
  of much of Atewa's real mining pressure (worked along river valleys,
  not through wholesale clear-felling).
- **Cross-reference `CLASS_SUBCANOPY_DISTURBANCE` pixel locations
  against real river valleys within the AOI** — alluvial mining
  concentrates along waterways by its real, physical nature (Hayakawa
  et al.'s own real study design explicitly buffers distance-to-water
  for this reason) — a spatial pattern following real drainage lines
  would be a strong, real corroborating signal, not proof on its own.
- **A real, growing `pct_visible_disturbance`** would indicate mining
  (or other clearing) has progressed to outright canopy removal by the
  November 2023 post-date — the real, more advanced stage of
  disturbance optical alone can already detect.

## Honest limitations of this specific project

- The real SLC coregistration used here (`extract_pair`'s coarse
  sub-swath crop) is honestly weaker than the full, rigorous pipeline
  documented in [InSAR Processing](../processing/insar.md) — a
  genuinely trustworthy real coherence result needs that fuller chain.
- `AOI` targets the specific real area named in a September 2026
  enforcement action — real mining pressure elsewhere in Atewa's real
  255 km² extent would need its own, separately-run AOI.
- Ten months (Jan-Nov 2023) is a real, reasonable window for this
  demonstration, but real alluvial mining activity can change on
  timescales of weeks — a real, operational monitoring use of this
  pipeline would run far more frequently.
